### Step 0 - Parameters and Imports


In [0]:
%run "../utils/config"

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import current_timestamp, desc, rank, count, when, col, sum


### Step 1 - Get the dfs

In [ ]:
race_results_df = spark.read.table("f1_presentation.race_results")

### Step 2 - Transform data

In [0]:
agg_df = race_results_df.groupBy("race_year", "team", "team_nationality").agg(
    sum("points").alias("total_points"),
    count(when(col("position") == 1, True)).alias("wins")
)

In [0]:
constructor_rank_spec = Window.partitionBy("race_year").orderBy(desc("total_points"), desc("wins"))

In [0]:
constructor_standings_df = (agg_df
                            .withColumn("rank",rank().over(constructor_rank_spec))
                            .withColumn("ingestion_date", current_timestamp())
                            )

### Step 3 - Write data

In [ ]:
constructor_standings_df.write.format("delta").mode("overwrite").saveAsTable("f1_presentation.constructor_standings")